## Preliminares

In [1]:
# Importar modulos y funciones necesarias
import pandas as pd
import numpy as np
from datetime import datetime
from statsmodels.formula.api import ols
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from funciones import *

In [2]:
# Abrir archivo clean_data
data_folder = "../data"
df = pd.read_parquet(f"{data_folder}/clean_data.parquet")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   YearsSinceAdded                 503 non-null    float64 
 1   Beta                            503 non-null    float64 
 2   DividendYield                   503 non-null    float64 
 3   ReturnOnAssets                  503 non-null    float64 
 4   profitMargins                   503 non-null    float64 
 5   operatingMargins                503 non-null    float64 
 6   currentRatio                    503 non-null    float64 
 7   revenueGrowth                   503 non-null    float64 
 8   shortPercentOfFloat             503 non-null    float64 
 9   PriceToBook_Transformed         503 non-null    float64 
 10  returnOnEquity_Transformed      503 non-null    float64 
 11  ForwardPE_Transformed           503 non-null    float64 
 12  MarketCap_log         

# Modelo inicial: Regresión Lineal

Se genera en primer lugar un modelo de regresion lineal a fin de comprobar la relevancia de las variables.

Se analizan primero las relaciones de variables continuas con la variable objetivo mediante la V de Cramer:

In [11]:
# Se generan 2 variables aleatorias para referencia
df['random1'] = np.random.uniform(0,1,size=df.shape[0])
df['random2'] = np.random.uniform(0,1,size=df.shape[0])

# Se separan las variables objetivo y los inputs de predictores
label = 'EnterpriseToEbitda_Transformed'
varObjCont = df[label]

imput = df.drop([label, 'Ticker'],axis=1)

# Ranking V de Cramer vs. obj continua
tablaCramer = pd.DataFrame(imput.apply(lambda x: cramers_v(x,varObjCont)),columns=['VCramer'])

px.bar(tablaCramer,x=tablaCramer.VCramer,title='Relaciones frente a la variable objetivo').update_yaxes(categoryorder="total ascending")

In [12]:
# Se eliminan random1 y random2 luego de contrastar su baja significancia en el modelo
df = df.drop(['random1', 'random2'], axis=1)

In [13]:
# Formula Primer Modelo OLS: Modelo Completo
form = ols_formula(df, label, 'Ticker')
form

'EnterpriseToEbitda_Transformed ~ YearsSinceAdded + Beta + DividendYield + ReturnOnAssets + profitMargins + operatingMargins + currentRatio + revenueGrowth + shortPercentOfFloat + PriceToBook_Transformed + returnOnEquity_Transformed + ForwardPE_Transformed + MarketCap_log + debtToEquity_log + trailingPegRatio_log + Sector + SubIndustry'

In [14]:
modelo1 = ols(form,data=df).fit()
print(modelo1.summary())

                                  OLS Regression Results                                  
Dep. Variable:     EnterpriseToEbitda_Transformed   R-squared:                       0.715
Model:                                        OLS   Adj. R-squared:                  0.599
Method:                             Least Squares   F-statistic:                     6.172
Date:                            Fri, 05 Jun 2026   Prob (F-statistic):           1.29e-44
Time:                                    16:52:06   Log-Likelihood:                -384.79
No. Observations:                             503   AIC:                             1062.
Df Residuals:                                 357   BIC:                             1678.
Df Model:                                     145                                         
Covariance Type:                        nonrobust                                         
                                                                             coef    std e

# Modelo de ensamblado de arboles RandomForest

In [15]:
# definir matrices de entrada y objetivo
X = df.drop([label,'Ticker'], axis=1)
y = df[label]

# identificar columnas numéricas y categóricas
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

# preprocesador: escala numéricas y codifica categóricas
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

pipe = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=25,
        min_samples_leaf=1,
        max_features='sqrt',
        max_samples= None        
        ))
])

pipe.fit(X_train, y_train)

# comparar predicciones con el conjunto test


y_pred = pipe.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MSE: 0.8270907877124375
R2: 0.34681028287414895


In [16]:
# Reajustar modelo con todos los datos
print("Reajustando modelo con el dataset completo...")

X_completo = df.drop([label,'Ticker'], axis=1)
y_completo = df[label]

pipe_final = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=3,
        min_samples_split=5,
        max_features=0.5,
        max_samples= 0.7        
        ))
])

pipe_final.fit(X_completo, y_completo)
r2_final = pipe_final.score(X_completo, y_completo)
print(f"R² sobre dataset completo: {r2_final:.4f}")

Reajustando modelo con el dataset completo...
R² sobre dataset completo: 0.8165


In [17]:
# Test de validación cruzada
cross_val_scores = cross_val_score(pipe_final, X_completo, y_completo, cv=5, scoring='r2')
print(f"R² promedio CV: {cross_val_scores.mean():.4f} ± {cross_val_scores.std():.4f}")

R² promedio CV: 0.6244 ± 0.1702


* Se detecta un gran sobre-ajuste del modelo, con la bondad de ajuste del modelo en Validacion Cruzada descendiendo desde 82% al 51%.
* Al ser una muestra pequena con solo 500 observaciones, se optimizan los hiperparametros reduciendo la maxima profundidad de los arboles, y aumentar el minimo de ejemplos por rama para evitar que el modelo *memorice* a cada accion del dataset.

In [18]:
# Importancia de factores en el modelo
rf_model = pipe_final.named_steps['model']
importances = rf_model.feature_importances_

# Obtener los nombres de las características después del preprocesamiento
preprocessor = pipe_final.named_steps['pre']
feature_names = preprocessor.get_feature_names_out()

feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
feature_importance_df.head(20)

,feature,importance
11,num__ForwardPE_Transformed,0.332804
7,num__revenueGrowth,0.127410
9,num__PriceToBook_Transformed,0.124988
1,num__Beta,0.080392
2,num__DividendYield,0.062227
13,num__debtToEquity_log,0.050651
12,num__MarketCap_log,0.040753
3,num__ReturnOnAssets,0.037261
10,num__returnOnEquity_Transformed,0.031151
8,num__shortPercentOfFloat,0.023476


## Aplicacion del modelo

Se generan 2 clusters segun las predicciones:
* Overprediction: si los residuos son mayores o iguales a cero.
*  Underprediction: si los residuos son menores a cero.

In [19]:
# Cluster: sobre/sub prediccion respecto a la identidad
residuos = y_pred - y_test
cluster = pd.Series(['Overprediction' if r >= 0 else 'Underprediction' for r in residuos], 
                     index=y_test.index)

# Visualizar con los 2 clusters
fig = px.scatter(x=y_test, y=y_pred, color=cluster,
                 labels={'x':'Real','y':'Predicho','color':'Cluster'},
                 title='Predicciones vs Reales (Clusters)')
fig.add_shape(type='line', x0=y_test.min(), y0=y_test.min(),
             x1=y_test.max(), y1=y_test.max(),
             line=dict(color='black', dash='dash', width=2))
fig.show()

# Estadísticas por cluster
print("\nEstadísticas por cluster:")
print(f"Overprediction: {(cluster == 'Overprediction').sum()} casos, residuo medio: {residuos[cluster == 'Overprediction'].mean():.4f}")
print(f"Underprediction: {(cluster == 'Underprediction').sum()} casos, residuo medio: {residuos[cluster == 'Underprediction'].mean():.4f}")


Estadísticas por cluster:
Overprediction: 79 casos, residuo medio: 0.4134
Underprediction: 47 casos, residuo medio: -0.4312


In [21]:
# Recrear clusters con predicciones del modelo final
y_pred_completo = pipe_final.predict(X_completo)
residuos_completo = y_pred_completo - y_completo
cluster_completo = pd.Series(['Overprediction' if r > 0 else 'Underprediction' for r in residuos_completo], 
                              index=y_completo.index)

# Actualizar columna de cluster y calcular distancia a la identidad
df['cluster_prediction'] = cluster_completo.values
# Distancia perpendicular a la recta identidad (y = x)
df['distance_identity'] = residuos_completo / np.sqrt(2)

# Visualizar con plotly
fig = px.scatter(df, x=label, y=y_pred_completo, color='cluster_prediction',
                 hover_data=['Ticker','Sector'],
                 labels={'x':'Label (Real)','y':'Label (Predicho)','cluster_prediction':'Cluster'},
                 title='Clusters Overprediction/Underprediction (Datos Completos)')
fig.add_shape(type='line', x0=y_completo.min(), y0=y_completo.min(),
             x1=y_completo.max(), y1=y_completo.max(),
             line=dict(color='black', dash='dash', width=2))
fig.show()

# Estadísticas por cluster
print("\nEstadísticas de clusters (datos completos):")
print(f"Overprediction: {(df['cluster_prediction'] == 'Overprediction').sum()} casos, residuo medio: {residuos_completo[cluster_completo == 'Overprediction'].mean():.4f}")
print(f"Underprediction: {(df['cluster_prediction'] == 'Underprediction').sum()} casos, residuo medio: {residuos_completo[cluster_completo == 'Underprediction'].mean():.4f}")


Estadísticas de clusters (datos completos):
Overprediction: 283 casos, residuo medio: 0.1816
Underprediction: 220 casos, residuo medio: -0.2355


In [22]:
print("\nDataframe con cluster:")
print(df[['Ticker', label, 'distance_identity','cluster_prediction']].sort_values('distance_identity', ascending=False).head(10))


Dataframe con cluster:
    Ticker  EnterpriseToEbitda_Transformed  distance_identity  \
69      BA                       -6.330856           4.331884   
317   MRNA                       -2.108437           1.061491   
61   BRK-B                       -1.712358           1.019099   
88    CVNA                        0.336532           0.755535   
292      L                       -0.868079           0.607085   
241    HUM                       -0.660785           0.543932   
318    MOH                       -0.900376           0.540303   
446    TKO                       -0.077946           0.472072   
263    JBL                       -0.019347           0.452003   
476    VST                       -0.610638           0.381450   

    cluster_prediction  
69      Overprediction  
317     Overprediction  
61      Overprediction  
88      Overprediction  
292     Overprediction  
241     Overprediction  
318     Overprediction  
446     Overprediction  
263     Overprediction  
476     Ov

In [23]:
# Guardar dataframe con cluster con fecha en el nombre del archivo

dia = datetime.now().day
mes = datetime.now().month
year = datetime.now().year

df.to_csv(f"{data_folder}/cluster_data_{year}_{mes}_{dia}.csv", index=False)

## Anexo: optimizacion de hiperparametros

* La siguiente celda se utilizó para optimizar los hiperparámetros de los modelos de forma individual. La ejecución de la misma no es necesaria.
* En caso de utilizarla, reemplazar 'no' por 'si' en la primer línea. Puede demorar varios minutos.

In [22]:
ejecutar_celda = 'si' # Reemplazar para ejecutar
modelo_a_optimizar = 1  # 1 = Random Forest 

if ejecutar_celda == 'si':

    if modelo_a_optimizar == 1:
        nombre_modelo= 'Random Forest'
        print(f"Configurando GridSearchCV para {nombre_modelo}")

        # Pipeline usando el preprocesador específico para Random Forest
        modelo_base = Pipeline(steps=[
            ('preprocesador', preprocessor),
            ('rf', RandomForestRegressor(random_state=42))
        ])

        param_grid = {
            'rf__n_estimators': [300],
            'rf__max_depth': [10],
            'rf__min_samples_leaf': [3],
            'rf__min_samples_split': [5],
            'rf__max_features': ['sqrt', 'log2', 0.5],
            'rf__max_samples': [0.7]
        }

    elif modelo_a_optimizar == 2: # Por implementar
        nombre_modelo= 'LightGBM'
        print(f"Configurando GridSearchCV para {nombre_modelo}")

        # Pipeline usando el preprocesador específico para LightGBM
        modelo_base = Pipeline(steps=[
            ('preprocesador', preprocesado_rf),
            ('lgb', lgb.LGBMRegressor(random_state=42,
                                      subsample=0.8,
                                      colsample_bytree=0.8
))
        ])

        param_grid = {
            'lgb__n_estimators': [500, 550, 600],
            'lgb__max_depth': [5],
            'lgb__learning_rate': [0.05],
            'lgb__num_leaves': [20],
            'lgb__min_child_samples': [20]
        }

    else:
        raise ValueError("Opción no válida. Por favor, selecciona 1 o 2.")

    # Configurar el GridSearchCV
    grid_search = GridSearchCV(
        estimator=modelo_base,
        param_grid=param_grid,
        scoring='neg_mean_absolute_error',
        cv=3,
        n_jobs=-1,
        verbose=2
    )

    # Entrenar con X_train y y_train originales
    print(f"Iniciando búsqueda de hiperparámetros. Esto tomará unos minutos.")
    grid_search.fit(X_train, y_train)

    # Resultados
    print("\n--- Búsqueda Finalizada ---")
    print("Mejores hiperparámetros encontrados:")
    print(grid_search.best_params_)

    # Extraer el mejor modelo
    best_model = grid_search.best_estimator_

    # Predecir en el set de prueba usando X_test original
    y_pred = best_model.predict(X_test)

    print("\nEvaluación en el set de Prueba:\n", obtener_metricas(y_test, y_pred, nombre_modelo))

Configurando GridSearchCV para Random Forest
Iniciando búsqueda de hiperparámetros. Esto tomará unos minutos.
Fitting 3 folds for each of 3 candidates, totalling 9 fits

--- Búsqueda Finalizada ---
Mejores hiperparámetros encontrados:
{'rf__max_depth': 10, 'rf__max_features': 0.5, 'rf__max_samples': 0.7, 'rf__min_samples_leaf': 3, 'rf__min_samples_split': 5, 'rf__n_estimators': 300}

Evaluación en el set de Prueba:
 {'Modelo': 'Random Forest', 'MAE': 0.5987146288314779, 'MSE': 0.6021282849392283, 'RMSE': np.float64(0.775969255150762), 'R2': 0.6147562531632547}
